# 03: Embedding Model Comparison

Compare different embedding models (Gemini API vs sentence-transformers) and understand their trade-offs.

## Prerequisites

- Python 3.10+
- `openai`, `sentence-transformers`, `numpy`, `scikit-learn`, `matplotlib`
- A OpenAI API key

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
import openai
import numpy as np
import time

load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
API_KEY = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=API_KEY)

print("Ready.")

## 2. Loading Models

In [ ]:
# Gemini embedding model
GEMINI_MODEL = "models/text-embedding-3-small"

# Sentence-transformers (local models)
from sentence_transformers import SentenceTransformer

MINILM_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
MPNET_MODEL = SentenceTransformer("all-mpnet-base-v2")

print(f"Gemini: {GEMINI_MODEL} (768 dims)")
print(f"MiniLM: all-MiniLM-L6-v2 (384 dims)")
print(f"MPNet:  all-mpnet-base-v2 (768 dims)")

## 3. Generate Embeddings with Each Model

In [ ]:
texts = [
    "What is machine learning?",
    "How does deep learning work?",
    "Explain neural networks",
    "What are transformers in AI?",
    "What is the weather today?",
    "Will it rain tomorrow?",
    "How do I cook pasta?",
    "Best recipes for dinner",
    "How to invest in stocks?",
    "Stock market analysis",
    "Learn Python programming",
    "Python tutorial for beginners",
]

# Gemini embeddings
gemini_result = genai.embed_content(model=GEMINI_MODEL, content=texts)
gemini_embeddings = np.array(gemini_result["embedding"])

# MiniLM embeddings
minilm_embeddings = MINILM_MODEL.encode(texts, normalize_embeddings=True)

# MPNet embeddings
mpnet_embeddings = MPNET_MODEL.encode(texts, normalize_embeddings=True)

print(f"Gemini:  {gemini_embeddings.shape}")
print(f"MiniLM:  {minilm_embeddings.shape}")
print(f"MPNet:   {mpnet_embeddings.shape}")

## 4. Benchmarking: Speed Comparison

In [ ]:
# Benchmark each model
test_texts = [f"This is test sentence number {i} for benchmarking." for i in range(50)]

# Gemini
start = time.time()
genai.embed_content(model=GEMINI_MODEL, content=test_texts)
gemini_time = time.time() - start

# MiniLM
start = time.time()
MINILM_MODEL.encode(test_texts, normalize_embeddings=True)
minilm_time = time.time() - start

# MPNet
start = time.time()
MPNET_MODEL.encode(test_texts, normalize_embeddings=True)
mpnet_time = time.time() - start

print(f"50 documents:")
print(f"  Gemini:  {gemini_time:.2f}s ({gemini_time/50*1000:.1f}ms/doc)")
print(f"  MiniLM:  {minilm_time:.2f}s ({minilm_time/50*1000:.1f}ms/doc)")
print(f"  MPNet:   {mpnet_time:.2f}s ({mpnet_time/50*1000:.1f}ms/doc)")
print(f"\nMiniLM is {gemini_time/minilm_time:.1f}x faster than Gemini (local vs API)")

## 5. Embedding Quality Comparison

Test how well each model captures semantic similarity.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


# Test pairs with expected similarity levels
test_pairs = [
    # High similarity (same topic)
    ("I love programming", "I enjoy coding", "High"),
    ("The stock market crashed", "Financial markets declined sharply", "High"),
    ("What is Python?", "Explain the Python language", "High"),
    # Medium similarity (related topics)
    ("Learn to code", "Learn data science", "Medium"),
    ("Python programming", "Java programming", "Medium"),
    # Low similarity (unrelated)
    ("I love programming", "The cat sat on the mat", "Low"),
    ("Stock market analysis", "How to cook pasta", "Low"),
]

print(f"{'Pair':<50} {'Gemini':<10} {'MiniLM':<10} {'MPNet':<10} {'Expected'}")
print("-" * 100)

for text_a, text_b, expected in test_pairs:
    # Gemini
    emb_a = genai.embed_content(model=GEMINI_MODEL, content=text_a)["embedding"]
    emb_b = genai.embed_content(model=GEMINI_MODEL, content=text_b)["embedding"]
    gemini_sim = cosine_similarity(np.array(emb_a), np.array(emb_b))
    
    # MiniLM
    emb_a = MINILM_MODEL.encode([text_a], normalize_embeddings=True)[0]
    emb_b = MINILM_MODEL.encode([text_b], normalize_embeddings=True)[0]
    minilm_sim = cosine_similarity(emb_a, emb_b)
    
    # MPNet
    emb_a = MPNET_MODEL.encode([text_a], normalize_embeddings=True)[0]
    emb_b = MPNET_MODEL.encode([text_b], normalize_embeddings=True)[0]
    mpnet_sim = cosine_similarity(emb_a, emb_b)
    
    pair_str = f"{text_a[:25]}... / {text_b[:20]}..."
    print(f"{pair_str:<50} {gemini_sim:<10.4f} {minilm_sim:<10.4f} {mpnet_sim:<10.4f} {expected}")

### Analysis

Which model:
1. Best distinguishes high vs low similarity pairs?
2. Most consistent in ranking?
3. Best for your use case?

## 6. Visualizing Embedding Clusters

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

labels = [
    "AI", "AI", "AI", "AI",
    "Weather", "Weather",
    "Food", "Food",
    "Finance", "Finance",
    "Programming", "Programming",
]

colors = {
    "AI": "#FF6B6B",
    "Weather": "#4ECDC4",
    "Food": "#45B7D1",
    "Finance": "#96CEB4",
    "Programming": "#FFEAA7",
}

# PCA for each model
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

models = [
    ("Gemini (768d)", gemini_embeddings),
    ("MiniLM (384d)", minilm_embeddings),
    ("MPNet (768d)", mpnet_embeddings),
]

for ax, (name, embeds) in zip(axes, models):
    pca = PCA(n_components=2)
    embeds_2d = pca.fit_transform(embeds)
    
    for label in set(labels):
        mask = [l == label for l in labels]
        ax.scatter(
            embeds_2d[mask, 0],
            embeds_2d[mask, 1],
            c=colors[label],
            label=label,
            s=100,
            alpha=0.7,
        )
    
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Building a Semantic Search with Each Model

Compare search quality across models.

In [ ]:
# Create search engines for each model
search_docs = [
    "Python is a popular programming language for data science.",
    "Machine learning models need large datasets for training.",
    "The weather forecast predicts rain this weekend.",
    "Neural networks are inspired by the human brain.",
    "FastAPI is a modern Python web framework.",
    "Climate change affects global weather patterns.",
    "Deep learning has revolutionized computer vision.",
    "JavaScript is essential for web development.",
]

# Gemini search
gemini_doc_result = genai.embed_content(model=GEMINI_MODEL, content=search_docs)
gemini_doc_embs = np.array(gemini_doc_result["embedding"])

# MiniLM search
minilm_doc_embs = MINILM_MODEL.encode(search_docs, normalize_embeddings=True)

# MPNet search
mpnet_doc_embs = MPNET_MODEL.encode(search_docs, normalize_embeddings=True)


def search_with_model(query, doc_embs, docs, model_name):
    """Search using specified model."""
    if model_name == "gemini":
        query_emb = np.array(genai.embed_content(model=GEMINI_MODEL, content=query)["embedding"])
    elif model_name == "minilm":
        query_emb = MINILM_MODEL.encode([query], normalize_embeddings=True)[0]
    elif model_name == "mpnet":
        query_emb = MPNET_MODEL.encode([query], normalize_embeddings=True)[0]
    
    similarities = np.array([
        cosine_similarity(query_emb, doc_emb)
        for doc_emb in doc_embs
    ])
    
    top_indices = similarities.argsort()[::-1][:3]
    return [(docs[i], similarities[i]) for i in top_indices]

In [ ]:
query = "artificial intelligence"

print(f"Query: '{query}'\n")

for model_name in ["gemini", "minilm", "mpnet"]:
    print(f"\n{model_name.upper()}:")
    print("-" * 60)
    results = search_with_model(query, gemini_doc_embs, search_docs, model_name)
    # Use correct embeddings for each model
    if model_name == "gemini":
        results = search_with_model(query, gemini_doc_embs, search_docs, model_name)
    elif model_name == "minilm":
        results = search_with_model(query, minilm_doc_embs, search_docs, model_name)
    elif model_name == "mpnet":
        results = search_with_model(query, mpnet_doc_embs, search_docs, model_name)
    
    for doc, score in results:
        print(f"  [{score:.4f}] {doc[:60]}")

## 8. When to Use Each Model

In [ ]:
comparison_table = """
| Feature              | Gemini (004) | MiniLM | MPNet  |
|---------------------|--------------|--------|--------|
| Dimensions          | 768          | 384    | 768    |
| Speed (relative)    | Slow (API)   | Fast   | Medium |
| Accuracy            | High         | Medium | High   |
| Cost                | Free tier    | Free   | Free   |
| Privacy             | Cloud        | Local  | Local  |
| Offline support     | No           | Yes    | Yes    |
| Best for            | Prototyping  | Speed  | Quality|
| Production-ready    | Yes          | Yes    | Yes    |

Recommendations:
- Quick prototyping / small projects: Gemini (easy to use, good accuracy)
- High-throughput / low-latency: MiniLM (fastest, local)
- Best accuracy / production: MPNet or Gemini
- Privacy-sensitive: MiniLM or MPNet (local)
- No internet: MiniLM or MPNet (local)
"""

print(comparison_table)

## 9. Practical: Build a Model-Agnostic Search Engine

Create a search engine that works with any embedding model.

In [ ]:
class EmbeddingSearchEngine:
    """Model-agnostic semantic search engine."""
    
    def __init__(self, embed_fn):
        """
        Args:
            embed_fn: Function that takes a list of texts and returns embeddings (np.array)
        """
        self.embed_fn = embed_fn
        self.documents = []
        self.embeddings = None
    
    def add_documents(self, documents):
        self.documents.extend(documents)
        new_embeddings = self.embed_fn(documents)
        
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
        
        print(f"Indexed {len(documents)} docs. Total: {len(self.documents)}")
    
    def search(self, query, top_k=3):
        query_embedding = self.embed_fn([query])
        
        similarities = np.array([
            cosine_similarity(query_embedding[0], doc_emb)
            for doc_emb in self.embeddings
        ])
        
        top_indices = similarities.argsort()[::-1][:top_k]
        return [(self.documents[i], similarities[i]) for i in top_indices]

In [ ]:
# Create search engines with different models
def gemini_embed(texts):
    result = genai.embed_content(model=GEMINI_MODEL, content=texts)
    return np.array(result["embedding"])

def minilm_embed(texts):
    return MINILM_MODEL.encode(texts, normalize_embeddings=True)

def mpnet_embed(texts):
    return MPNET_MODEL.encode(texts, normalize_embeddings=True)


# Initialize engines
gemini_engine = EmbeddingSearchEngine(gemini_embed)
minilm_engine = EmbeddingSearchEngine(minilm_embed)
mpnet_engine = EmbeddingSearchEngine(mpnet_embed)

# Same documents for fair comparison
docs = [
    "Python is a popular programming language for data science.",
    "Machine learning models need large datasets for training.",
    "The weather forecast predicts rain this weekend.",
    "Neural networks are inspired by the human brain.",
    "FastAPI is a modern Python web framework.",
    "Climate change affects global weather patterns.",
    "Deep learning has revolutionized computer vision.",
    "JavaScript is essential for web development.",
]

gemini_engine.add_documents(docs)
minilm_engine.add_documents(docs)
mpnet_engine.add_documents(docs)

In [ ]:
query = "web development"

print(f"Query: '{query}'\n")

for name, engine in [("Gemini", gemini_engine), ("MiniLM", minilm_engine), ("MPNet", mpnet_engine)]:
    print(f"{name}:")
    for doc, score in engine.search(query, top_k=3):
        print(f"  [{score:.4f}] {doc}")
    print()

## Exercises

1. **Custom evaluation**: Create 20 query-document pairs with known relevance scores. Compute precision@3 for each model. Which model performs best?

2. **Dimension reduction**: Use MiniLM with 384 dimensions. Compare search quality to Gemini with 768 dimensions. Does reducing dimensions by 50% hurt quality?

3. **Multilingual test**: Find a multilingual embedding model (e.g., `paraphrase-multilingual-MiniLM-L12-v2`). Test if it can match English queries to non-English documents.

4. **Production decision**: You're building a search engine for 1 million documents. Which model do you choose and why? Write a 1-paragraph justification.

## Summary

- Gemini API: Best accuracy, easy to use, requires internet
- MiniLM: Fastest, lightweight, good for speed-critical applications
- MPNet: Best balance of speed and accuracy for local deployment
- Model choice depends on: accuracy needs, latency requirements, cost, and privacy constraints
- All models can build effective semantic search systems

→ Next: [Exercise 01](../exercises/exercise-01.md)